In [ ]:
!pip install transformers accelerate bitsandbytes datasets peft

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Choose an alternative model if needed
model_name = "TinyLlama/TinyLlama-1.1B-step-50K"  # Replace with another model if needed

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model with 8-bit quantization for efficiency
model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    device_map="auto", 
    load_in_8bit=True
)

OSError: TinyLlama/TinyLlama-1.1B-step-50K is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`

In [ ]:
from datasets import Dataset

data = [
    {"question": "If it takes 1 hour for 60 people to play an Opera, how many hours will it take 600 people to play the same opera?", 
     "answer": "1 hour. More people don't change the duration of the opera."},

    {"question": "Is a pound of feathers or a British pound heavier?", 
     "answer": "A British pound is currency, while a pound of feathers is a unit of weight. The feathers weigh more."},

    {"question": "A boy runs down the stairs in the morning and sees a tree in his living room, and some boxes under the tree. What's going on?", 
     "answer": "It is likely Christmas morning, and the boy sees a Christmas tree with presents underneath."},

    {"question": "What happens if you crack your knuckles a lot?", 
     "answer": "Cracking knuckles releases gas from the joints. It does not cause arthritis but may lead to reduced grip strength."},

    {"question": "If there is a shark in the pool of my basement, is it safe to go upstairs?", 
     "answer": "Yes, because the shark is in the pool and not upstairs."},

    {"question": "How much wood could a woodchuck chuck if there were only 5 pounds of wood in the world?", 
     "answer": "A woodchuck could only chuck up to 5 pounds of wood in this case."},

    {"question": "Who is the current President of the United States?", 
     "answer": "As of 2025, the President of the United States is [UPDATE THIS BASED ON REAL DATA]."},

    {"question": "Was Talos alive?", 
     "answer": "No, Talos was a mythical automaton from Greek mythology, not a living being."},

    {"question": "How many Ls are in the word parallel?", 
     "answer": "There are three Ls in the word 'parallel'."},

    {"question": "What is the riddle of the sphinx, and what are all possible answers satisfying all conditions?", 
     "answer": "The classic riddle is 'What walks on four legs in the morning, two in the afternoon, and three in the evening?' The answer is a human (crawling as a baby, walking as an adult, using a cane in old age)."}
]

dataset = Dataset.from_list(data)


In [ ]:
def tokenize_function(examples):
    prompt = [f"Q: {q} A: {a}" for q, a in zip(examples["question"], examples["answer"])]
    return tokenizer(prompt, padding="max_length", truncation=True, max_length=256)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

In [ ]:
from peft import get_peft_model, LoraConfig, TaskType
from transformers import Trainer, TrainingArguments

# Apply LoRA for efficient tuning
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, 
    inference_mode=False,
    r=8,  # Rank
    lora_alpha=16,
    lora_dropout=0.1
)

model = get_peft_model(model, peft_config)

# Training parameters
training_args = TrainingArguments(
    output_dir="./llama-finetuned",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    save_steps=100,
    save_total_limit=2,
    logging_steps=10,
    optim="adamw_bnb_8bit",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()

In [ ]:
def generate_answer(question):
    input_text = f"Q: {question} A:"
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to("cuda")
    
    output = model.generate(input_ids, max_length=100)
    return tokenizer.decode(output[0], skip_special_tokens=True)

question = "How many Ls are in the word parallel?"
print(generate_answer(question))